# Validação 09 — Ranking híbrido de evidências

## Goal

Comprovar que BM25 e embeddings podem ser combinados em um único ranking auditável sem somar diretamente scores de escalas incompatíveis.

## Setup

A fusão usa Weighted Reciprocal Rank Fusion (RRF). Cada método contribui conforme a posição do trecho em seu ranking: `peso / (rrf_k + posição)`. O notebook usa pesos iguais e `rrf_k=60`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from fatofake import (
    DEFAULT_EMBEDDING_MODEL,
    Bm25Index,
    ChunkingConfig,
    HybridConfig,
    HybridIndex,
    PmcClient,
    PubMedClient,
    SemanticIndex,
    SentenceTransformerEncoder,
    chunk_article_content,
    prepare_search_plan,
    retrieve_article_content,
    search_pubmed,
    validate_analysis_input,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f"Execução UTC: {executed_at}")

Execução UTC: 2026-09-23T14:35:13.091029+00:00


## Steps

### 1. Recuperar e recortar a fonte

Reexecutamos o fluxo anterior para manter a validação independente de estado oculto.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ["33431520[pmid]"]

claim = "Beber café pode alterar o risco de câncer de próstata."
analysis_input = validate_analysis_input(claim)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_result = search_pubmed(
    search_plan,
    PubMedClient(email=os.getenv("NCBI_EMAIL"), api_key=os.getenv("NCBI_API_KEY")),
    max_results_per_query=1,
)
content = retrieve_article_content(
    pubmed_result.publications[0],
    PmcClient(email=os.getenv("NCBI_EMAIL"), api_key=os.getenv("NCBI_API_KEY")),
)
chunks = chunk_article_content(content, ChunkingConfig(max_words=120, overlap_words=20))
print(f"Fonte: {content.pmcid} | trechos: {len(chunks)} | alegação: {claim}")

Fonte: PMC7805365 | trechos: 35 | alegação: Beber café pode alterar o risco de câncer de próstata.


### 2. Construir e executar o ranking híbrido

O índice solicita mais candidatos do que o total final para que a fusão consiga reconhecer trechos relevantes que aparecem em posições diferentes.

In [3]:
top_k = 5
lexical_index = Bm25Index(chunks)
semantic_index = SemanticIndex(
    chunks,
    SentenceTransformerEncoder(
        os.getenv("EMBEDDING_MODEL", DEFAULT_EMBEDDING_MODEL)
    ),
)
hybrid_config = HybridConfig(
    lexical_weight=1.0,
    semantic_weight=1.0,
    rrf_k=60,
    candidate_multiplier=4,
)
hybrid_index = HybridIndex(lexical_index, semantic_index, hybrid_config)
hybrid_results = hybrid_index.search(claim, top_k=top_k)

print(f"Modelo semântico: {semantic_index.model_name}")
print(f"Resultados híbridos: {len(hybrid_results)}")

/private/tmp/fatofake-notebook-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23863.98it/s]

Modelo semântico: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Resultados híbridos: 5


In [4]:
ranking = [
    {
        "rank": result.rank,
        "rrf_score": round(result.score, 6),
        "bm25_rank": result.lexical_rank,
        "semantic_rank": result.semantic_rank,
        "bm25_contribution": round(result.lexical_contribution, 6),
        "semantic_contribution": round(result.semantic_contribution, 6),
        "section": result.chunk.section,
        "chunk_id": result.chunk.chunk_id,
        "preview": result.chunk.text[:260],
        "source_url": result.chunk.source_url,
    }
    for result in hybrid_results
]
pprint(ranking)

[{'bm25_contribution': 0.015385,
  'bm25_rank': 5,
  'chunk_id': '33431520:1:4:b46f6ee9c488ee7c',
  'preview': 'different subgroups.22 23 Since then, five cohort studies have '
             'explored the association but still reported inconsistent '
             'results.24–28 It was hypothesised that higher coffee consumption '
             'was associated with an increased risk of prostate cancer. Thus, '
             'the objecti',
  'rank': 1,
  'rrf_score': 0.031514,
  'section': 'Introduction',
  'semantic_contribution': 0.016129,
  'semantic_rank': 2,
  'source_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/'},
 {'bm25_contribution': 0.014925,
  'bm25_rank': 7,
  'chunk_id': '33431520:1:3:780ab10c7a21b35b',
  'preview': 'with the risk of prostate cancer. Although earlier cohort '
             'studies did not detect an association,7–15 more recent studies '
             'conducted in major Western countries such as the USA, Sweden and '
             'the UK reported tha

### 3. Comparar a cobertura dos métodos

A comparação abaixo usa apenas posições e presença nos rankings; os scores brutos de BM25 e cosseno permanecem separados.

In [5]:
lexical_top = lexical_index.search(claim, top_k=top_k)
semantic_top = semantic_index.search(claim, top_k=top_k)
lexical_ids = {result.chunk.chunk_id for result in lexical_top}
semantic_ids = {result.chunk.chunk_id for result in semantic_top}
hybrid_ids = {result.chunk.chunk_id for result in hybrid_results}

coverage = {
    "híbridos presentes no top 5 BM25": len(hybrid_ids & lexical_ids),
    "híbridos presentes no top 5 semântico": len(hybrid_ids & semantic_ids),
    "híbridos presentes nos dois top 5": len(hybrid_ids & lexical_ids & semantic_ids),
}
pprint(coverage)

{'híbridos presentes no top 5 BM25': 2,
 'híbridos presentes no top 5 semântico': 4,
 'híbridos presentes nos dois top 5': 1}


## Checks

As verificações confirmam a fórmula RRF, ordenação, repetibilidade, relevância mínima e proveniência.

In [6]:
assert len(hybrid_results) == top_k
assert [result.rank for result in hybrid_results] == list(range(1, top_k + 1))
assert all(
    previous.score >= current.score
    for previous, current in zip(hybrid_results, hybrid_results[1:])
)
assert len(hybrid_ids) == top_k
assert all(result.lexical_rank is not None or result.semantic_rank is not None for result in hybrid_results)
assert all(result.chunk.pmid == "33431520" for result in hybrid_results)
assert all(result.chunk.pmcid == "PMC7805365" for result in hybrid_results)
assert all(result.chunk.source_url == content.pmc_url for result in hybrid_results)
assert any("prostate cancer" in result.chunk.text.casefold() for result in hybrid_results)

for result in hybrid_results:
    expected_lexical = (
        hybrid_config.lexical_weight / (hybrid_config.rrf_k + result.lexical_rank)
        if result.lexical_rank is not None
        else 0.0
    )
    expected_semantic = (
        hybrid_config.semantic_weight / (hybrid_config.rrf_k + result.semantic_rank)
        if result.semantic_rank is not None
        else 0.0
    )
    assert abs(result.lexical_contribution - expected_lexical) < 1e-12
    assert abs(result.semantic_contribution - expected_semantic) < 1e-12
    assert abs(result.score - expected_lexical - expected_semantic) < 1e-12

second_run = hybrid_index.search(claim, top_k=top_k)
assert [result.chunk.chunk_id for result in hybrid_results] == [
    result.chunk.chunk_id for result in second_run
]

dual_signal_count = sum(
    result.lexical_rank is not None and result.semantic_rank is not None
    for result in hybrid_results
)
print(
    f"Validação aprovada: {top_k} resultados híbridos; "
    f"{dual_signal_count} receberam contribuição dos dois métodos."
)

Validação aprovada: 5 resultados híbridos; 5 receberam contribuição dos dois métodos.


## Next Steps

O ranking híbrido estará validado quando todas as células forem executadas sem erros. A próxima etapa será extrair afirmações verificáveis dos trechos recuperados e estruturar a comparação com a alegação do usuário.